# Deploy a SageMaker AI Recommendation (boto3)

This notebook deploys a recommendation produced by the **SageMaker AI Recommendation Service**. The recommendation service outputs a **Model Package** containing one or more `InferenceSpecifications` — each one is an instance-specific deployment configuration that may include optimizations like speculative decoding (EAGLE 3), quantization, or kernel tuning.

To deploy, the Model Package must first be converted into a **Deployable Model** via `CreateModel`, then deployed as a SageMaker endpoint.

**Workflow:**
1. Fetch the recommendation and extract `ModelPackageArn` and `InferenceSpecificationName`
2. Convert the Model Package into a Deployable Model (`CreateModel`)
3. Create an endpoint config and deploy the endpoint
4. Run a test inference
5. Clean up

**Prerequisites:**
- A completed AI Recommendation Job (status: `Completed`)
- An IAM execution role with SageMaker and S3 permissions
- Sufficient instance quota for the recommended instance type

## Step 1 — Install dependencies

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

## Step 2 — Configuration

Set the recommendation job name. `ROLE_ARN` is automatically retrieved from the SageMaker execution role attached to this Studio environment. All subsequent cells use these variables.

In [ ]:
import boto3
import json
import time
from datetime import datetime

def get_execution_role():
    sts = boto3.client('sts')
    arn = sts.get_caller_identity()['Arn']
    if ':assumed-role/' in arn:
        parts = arn.split(':')
        account = parts[4]
        role_name = parts[5].split('/')[1]
        return f"arn:aws:iam::{account}:role/{role_name}"
    raise RuntimeError(f"Cannot derive execution role from ARN: {arn}. Set ROLE_ARN manually.")

REGION                  = "us-west-2"  # <-- change if deploying in a different region
RECOMMENDATION_JOB_NAME = "<your-recommendation-job-name>"  # replace with your job name
ENDPOINT_NAME           = f"rec-endpoint-{datetime.now().strftime('%Y%m%d%H%M%S')}"
WAIT_FOR_READY          = True   # block until endpoint is InService

sm         = boto3.client("sagemaker", region_name=REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=REGION)
ROLE_ARN   = get_execution_role()

print(f"Role ARN      : {ROLE_ARN}")
print(f"Endpoint name : {ENDPOINT_NAME}")
print(f"Region        : {REGION}")

### Ensure role trust policy allows SageMaker

Checks the trust policy of the execution role and adds `sagemaker.amazonaws.com` if missing. This is a no-op if the trust policy is already correct.

In [ ]:
iam = boto3.client('iam')
role_name = ROLE_ARN.split('/')[-1]

trust = iam.get_role(RoleName=role_name)['Role']['AssumeRolePolicyDocument']
principals = [
    s.get('Principal', {}).get('Service', '')
    for s in trust.get('Statement', [])
]
flat = [p for item in principals for p in (item if isinstance(item, list) else [item])]

if 'sagemaker.amazonaws.com' in flat:
    print('✅ Trust policy already allows sagemaker.amazonaws.com')
else:
    trust['Statement'].append({
        'Effect': 'Allow',
        'Principal': {'Service': 'sagemaker.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    })
    iam.update_assume_role_policy(
        RoleName=role_name,
        PolicyDocument=json.dumps(trust)
    )
    print(f'✅ Added sagemaker.amazonaws.com to trust policy for role: {role_name}')

## Step 3 — Fetch the recommendation

Retrieves the first recommendation from the completed job. Each recommendation contains:
- **`ModelDetails.ModelPackageArn`** — the Model Package produced by the recommendation service
- **`ModelDetails.InferenceSpecificationName`** — the specific instance-optimized config within that package
- **`DeploymentConfiguration`** — container image, instance type, copy count, and environment variables
- **`OptimizationDetails`** — optimizations applied (e.g. `SpeculativeDecoding`, `Quantization`)
- **`ExpectedPerformance`** — predicted latency, throughput, and TTFT

If the job returned multiple recommendations, they are ordered by the service — the first is the top recommendation for your performance target.

In [ ]:
desc = sm.describe_ai_recommendation_job(AIRecommendationJobName=RECOMMENDATION_JOB_NAME)

assert desc["AIRecommendationJobStatus"] == "Completed", \
    f"Job is not completed yet. Status: {desc['AIRecommendationJobStatus']}"

# Take the top recommendation
rec = desc["Recommendations"][0]

model_package_arn          = rec["ModelDetails"]["ModelPackageArn"]
inference_spec_name        = rec["ModelDetails"]["InferenceSpecificationName"]
dc                         = rec["DeploymentConfiguration"]
instance_type              = dc["InstanceType"]
instance_count             = dc.get("InstanceCount", 1)
copy_count                 = dc.get("CopyCountPerInstance", 1)
env_vars                   = dc.get("EnvironmentVariables", {})

print(f"Model Package ARN          : {model_package_arn}")
print(f"Inference Specification    : {inference_spec_name}")
print(f"Instance type              : {instance_type}")
print(f"Instance count             : {instance_count}")
print(f"Copies per instance        : {copy_count}")
print(f"\nOptimizations applied:")
for opt in rec.get("OptimizationDetails", []):
    print(f"  - {opt['OptimizationType']}")
print(f"\nExpected performance:")
for perf in rec.get("ExpectedPerformance", []):
    stat = f" ({perf['Stat']})".rstrip(" ()")
    print(f"  {perf['Metric']}{stat}: {perf['Value']} {perf.get('Unit', '')}")

## Step 4 — Convert Model Package to a Deployable Model

The recommendation service outputs a **Model Package** with one or more `InferenceSpecifications`. To deploy on SageMaker Hosting, the Model Package must first be converted into a **Deployable Model** using `CreateModel`.

Pass `ModelPackageName` (the ARN) and `InferenceSpecificationName` to select the specific instance-optimized configuration. If the recommendation used `CopyCountPerInstance > 1` (e.g. 8 copies on a p5en.48xlarge), set `SM_NUM_ACCELERATORS` so each copy claims only its share of GPUs.

In [ ]:
model_name = f"rec-model-{datetime.now().strftime('%Y%m%d%H%M%S')}"

# If multiple copies share the instance, each copy gets its own GPU slice
merged_env = dict(env_vars)
# Detect tensor parallel size — vLLM uses SM_VLLM_ prefix, LMI uses OPTION_ prefix
# If neither is set (e.g. LMI without explicit TP config), default to 1
tp_size = int(
    merged_env.get("SM_VLLM_TENSOR_PARALLEL_SIZE")      # vLLM container
    or merged_env.get("OPTION_TENSOR_PARALLEL_DEGREE")  # LMI container
    or 1
)
if copy_count > 1:
    merged_env["SM_NUM_ACCELERATORS"] = str(tp_size)

sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=ROLE_ARN,
    # Convert Model Package → Deployable Model by referencing the
    # ModelPackageArn and the specific InferenceSpecificationName
    Containers=[
        {
            "ModelPackageName": model_package_arn,
        }
    ],
    PrimaryContainer=None,  # not used when deploying from a Model Package
)

print(f"Deployable model created: {model_name}")

## Step 5 — Create endpoint config and deploy

Creates the endpoint config using the recommended instance type and count, then deploys the endpoint.

For deployments with `CopyCountPerInstance > 1`, set `ModelDataDownloadTimeout` and `ContainerStartupHealthCheckTimeout` to allow time for all copies to load.

In [ ]:
endpoint_config_name = f"{ENDPOINT_NAME}-config"

variant = {
    "VariantName"        : "AllTraffic",
    "ModelName"          : model_name,
    "InstanceType"       : instance_type,
    "InitialInstanceCount": instance_count,
}

# Allow extra startup time when multiple model copies share the instance
if copy_count > 1:
    variant["ModelDataDownloadTimeoutInSeconds"]            = 3600
    variant["ContainerStartupHealthCheckTimeoutInSeconds"]  = 600

sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[variant],
)

sm.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=endpoint_config_name,
)

print(f"Endpoint creation started: {ENDPOINT_NAME}")

## Step 6 — Wait for endpoint to be InService

Polls every 30 seconds until the endpoint reaches `InService` status. Large models on multi-GPU instances may take several minutes to load.

In [ ]:
if WAIT_FOR_READY:
    print(f"Waiting for endpoint '{ENDPOINT_NAME}'...")
    while True:
        status = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
        print(f"  [{datetime.now().strftime('%H:%M:%S')}] {status}")
        if status == "InService":
            print("✅ Endpoint is InService")
            break
        if status in ("Failed", "OutOfService", "RollingBack"):
            raise RuntimeError(f"Endpoint failed with status: {status}")
        time.sleep(30)

## Step 7 — Test inference

Sends a test request to verify the endpoint is responding correctly. The endpoint uses the OpenAI-compatible chat completions API.

In [ ]:
payload = {
    "messages": [
        {"role": "user", "content": "What is the capital of France?"}
    ],
    "max_tokens": 128,
    "temperature": 0.1,
}

start = time.time()
response = sm_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload),
)
result = json.loads(response["Body"].read())
elapsed = time.time() - start

print(f"✅ Response time: {elapsed:.2f}s")
print(f"\nResponse:\n{result['choices'][0]['message']['content']}")
print(f"\nUsage: {result.get('usage', {})}")

## (Optional) Step 8 — Deploy as Inference Component

Alternatively, deploy the model as an **Inference Component** on a pre-existing empty endpoint. This is useful when you want to host multiple models on the same endpoint or control GPU allocation per model copy.

Uncomment and run this cell instead of Steps 5–6 if you prefer the Inference Component pattern.

In [ ]:
# PRECONDITION: an empty endpoint must already exist
# EXISTING_ENDPOINT_NAME = "my-precreated-empty-endpoint"
# ic_name = f"ic-{model_name}"
#
# sm.create_inference_component(
#     InferenceComponentName=ic_name,
#     EndpointName=EXISTING_ENDPOINT_NAME,
#     VariantName="AllTraffic",
#     Specification={
#         "ModelName": model_name,
#         "StartupParameters": {
#             "ModelDataDownloadTimeoutInSeconds": 3600,
#             "ContainerStartupHealthCheckTimeoutInSeconds": 600,
#         },
#         "ComputeResourceRequirements": {
#             "NumberOfAcceleratorDevicesRequired": tp_size,
#             "MinMemoryRequiredInMb": 10 * 1024,
#         },
#     },
#     RuntimeConfig={"CopyCount": copy_count},
# )
# print(f"Inference Component created: {ic_name}")

## Step 9 — Cleanup

Delete the endpoint, endpoint config, and model to avoid ongoing charges.

In [ ]:
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_model(ModelName=model_name)
print("Cleanup complete.")